# Klasifikasi DemogPairs Menggunakan ViT (Wajah, Emosi, dan Umur) & Random Forest

In [ ]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [ ]:
data = u.load_demogpairs()
pd.DataFrame(data)

## Load Fitur

In [ ]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

## Split Data

In [ ]:
X = np.array([features[d['image_path']] for d in data])y = np.array([d['label_idx'] for d in data])X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)print((len(X_train), len(X_test)))

## Kombinasi Parameter

In [ ]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

## Klasifikasi

In [ ]:
evaluation_results, fold_results = u.evaluate_models(grid_models, X_train, y_train, X_test, y_test, model_prefix='models/clf_demogpairs_rf_vit-face-emotion-age_', results_path='results/demogpairs_rf_vit-face-emotion-age_')sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')u.html_br()_dtable = u.display_table(sorted_results)

In [ ]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl')u.h(5, 'Waktu Pelatihan (Jobs)')u.seconds_to_time(round(training_time))

In [ ]:
u.h(5, 'Waktu Pelatihan')times = [fr['Train Time Mean'] * 5 for fr in fold_results]u.seconds_to_time(round(np.sum(times) + model.refit_time_))

In [ ]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])